# rotation-matrix-3d-y-axis — worked example 3: Rotate a point cloud at multiple angles and track one point

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `rotation-matrix-3d-y-axis`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import math
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When you have a batch of N 3D points stored as an (N, 3) tensor, you can rotate all of them in one operation using `points @ R.T`, where R is the 3×3 rotation matrix. Using R.T instead of R is necessary because the points are row vectors (shape (N, 3)), so the matrix must be transposed to get the right multiplication shape.

## Worked solution

**Step 1 — Create a small point cloud.** We build 5 points representing the vertices of a simple shape in the X-Z plane, all with Y=0 for clarity.

**Step 2 — Build R_y(θ) for a chosen angle.** We pick θ = 45° so the rotation is non-trivial but easy to reason about.

**Step 3 — Rotate the whole batch in one shot.** `points @ R.T` gives shape (5, 3). Each row i of the result is R applied to points[i].

**Step 4 — Verify one point manually.** The point (1, 0, 0) rotated 45° around Y should become (cos45°, 0, -sin45°) ≈ (0.707, 0, -0.707). We print this specific row and compare.

**Step 5 — Check that Y coordinates are preserved.** All points have Y=0, and rotation about Y doesn't change Y coordinates, so the Y column of the output should remain all zeros.

In [ ]:
import torch as t
import math

t.manual_seed(3)

def make_ry(deg: float) -> t.Tensor:
    theta = math.radians(deg)
    c, s = math.cos(theta), math.sin(theta)
    return t.tensor([[c, 0.0, s], [0.0, 1.0, 0.0], [-s, 0.0, c]], dtype=t.float32)

# 5 points in the X-Z plane (Y = 0)
points = t.tensor([
    [1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0],
    [1.0, 0.0, 1.0],
    [2.0, 0.0, 0.0],
    [0.5, 0.0, 0.5],
], dtype=t.float32)

angle = 45.0
R = make_ry(angle)
rotated = points @ R.T  # (5, 3)

print(f'Rotating {points.shape} batch by {angle}° around Y axis')
print('Rotated shape:', rotated.shape)
print('Original point (1,0,0) -> ', rotated[0].tolist())  # expect ~(0.707, 0, -0.707)
c45 = math.cos(math.radians(45))
print(f'Expected: ({c45:.3f}, 0.000, {-c45:.3f})')
print('Y column (should be all zeros):', rotated[:, 1].tolist())